In [4]:
!pip install numpy==1.26.4 pandas==1.5.3 scikit-learn==1.5.1 joblib==1.4.2 seaborn==0.13.2 --quiet

In [5]:
import json
import joblib
import time
import platform
import sklearn
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

output_dir = Path("models/penguins")
output_dir.mkdir(parents=True, exist_ok=True)

In [7]:
penguins = sns.load_dataset("penguins").dropna()

X = penguins[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]
y = penguins["species"]
target_names = y.unique().tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Shape entrenamiento:", X_train.shape)
print("Shape prueba:", X_test.shape)

Shape entrenamiento: (266, 4)
Shape prueba: (67, 4)


In [8]:
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, multi_class="multinomial"))
])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f" Accuracy en test: {acc:.4f}")
print("\nReporte de clasificación:\n")
print(classification_report(y_test, y_pred, target_names=target_names))

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


 Accuracy en test: 1.0000

Reporte de clasificación:

              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        29
   Chinstrap       1.00      1.00      1.00        14
      Gentoo       1.00      1.00      1.00        24

    accuracy                           1.00        67
   macro avg       1.00      1.00      1.00        67
weighted avg       1.00      1.00      1.00        67



In [9]:
model_path = output_dir / "penguins_model.pkl"
joblib.dump(pipeline, model_path)

meta = {
    "model_name": "penguins_logreg_pipeline",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "features": X.columns.tolist(),
    "target_names": target_names,
    "metrics": {"accuracy": acc},
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "pipeline": ["SimpleImputer(median)", "StandardScaler()", "LogisticRegression(max_iter=1000)"],
    "model_version": "1.0.0"
}
meta_path = output_dir / "penguins_meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f" Modelo guardado en: {model_path}")
print(f" Metadatos guardados en: {meta_path}")


 Modelo guardado en: models/penguins/penguins_model.pkl
 Metadatos guardados en: models/penguins/penguins_meta.json


In [10]:
sample = [[39.1, 18.7, 181.0, 3750.0]]  # Ejemplo de Adelie
pred_idx = pipeline.predict(sample)[0]
proba = pipeline.predict_proba(sample)[0]
print("Predicción (label):", pred_idx)
print("Probabilidades:", dict(zip(pipeline.classes_, proba)))

Predicción (label): Adelie
Probabilidades: {'Adelie': 0.9868967449426245, 'Chinstrap': 0.01290616404262563, 'Gentoo': 0.0001970910147499463}


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
